In [ ]:
# Cell 1: Environment Setup & Imports
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
# Cell 2: Sequence Generation Function
def create_sequences(df_ticker, seq_length):
    features = ['Close', 'Volume', 'Sentiment', 'SMA_20', 'RSI_14']
    target = 'Close'
    X, y = [], []
    for i in range(len(df_ticker) - seq_length):
        X.append(df_ticker[features].iloc[i : i + seq_length].values)
        y.append(df_ticker[target].iloc[i + seq_length])
    return np.array(X), np.array(y)


In [3]:
# Cell 3: BiLSTM + Attention Architecture
class StockPredictorBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1, dropout_rate=0.2):
        super(StockPredictorBiLSTMAttention, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_rate if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.attention = nn.Linear(hidden_size * 2, 1) # Weights bidirectional output
        self.fc1 = nn.Linear(hidden_size * 2, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context_vector = torch.sum(attn_weights * lstm_out, dim=1)
        out = self.dropout(context_vector)
        return self.fc2(self.relu(self.fc1(out)))


In [4]:
# Cell 4: Standardized Triple Loop (V2 BiLSTM + Attention)
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIGS ---
TICKERS = ['MSFT', 'DIS', 'WMT'] 
SEQ_LENGTHS = [5, 10, 20] # Triple testing
NUM_RUNS = 10
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.001

BASE_MODEL_DIR = "models/v2_attention"
BASE_RESULTS_DIR = "results/trainV2_attention"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for seq_len in SEQ_LENGTHS:
    print(f"\n" + "#"*60 + f"\nSTARTING V2 ATTN EXPERIMENTS | SEQUENCE LENGTH: {seq_len}\n" + "#"*60)
    SEQ_RESULTS = []
    
    for ticker in TICKERS:
        print(f"\n>>> Processing: {ticker} (Seq: {seq_len})")
        
        file_path = f"datasets/{ticker}/{ticker}_DatasetV1.csv"
        ticker_df = pd.read_csv(file_path)
        ticker_df['Trading_Date'] = pd.to_datetime(ticker_df['Trading_Date'])
        
        # Split by Year Consistent with V1
        train_raw = ticker_df[ticker_df['Trading_Date'].dt.year <= 2022].copy()
        test_raw = ticker_df[ticker_df['Trading_Date'].dt.year == 2023].copy()
        
        # Scaling: Fit ONLY on training data
        scaler = MinMaxScaler()
        cols = ['Close', 'Volume', 'Sentiment', 'SMA_20', 'RSI_14']
        train_raw[cols] = scaler.fit_transform(train_raw[cols])
        test_raw[cols] = scaler.transform(test_raw[cols])
        
        # Prepare Data
        X_train, y_train = create_sequences(train_raw, seq_len)
        X_test, y_test = create_sequences(test_raw, seq_len)
        
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        for run in range(1, NUM_RUNS + 1):
            # BiLSTM + Attention (V2 Architecture)
            model = StockPredictorBiLSTMAttention(input_size=5, hidden_size=128, num_layers=2).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            criterion = nn.MSELoss()
            
            train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
            
            model.train()
            for epoch in range(EPOCHS):
                for xb, yb in train_loader:
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()
            
            model.eval()
            with torch.no_grad():
                preds = model(X_test_t).cpu().numpy().flatten()
            
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)
            
            # Store Metrics
            SEQ_RESULTS.append({'Ticker': ticker, 'Run': run, 'Seq': seq_len, 'R2': r2, 'MAE': mae})
            
            # Save weights (.pth)
            save_path = os.path.join(BASE_MODEL_DIR, ticker)
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/v2_att_run_{run}_seq{seq_len}.pth")
            
            print(f"  Run {run:02d}: R2={r2:.4f}")

    # Save CSV Results
    results_df = pd.DataFrame(SEQ_RESULTS)
    csv_path = os.path.join(BASE_RESULTS_DIR, f"v2_att_stability_results_{seq_len}.csv")
    results_df.to_csv(csv_path, index=False)
    
    print(f"\n[DONE] Seq {seq_len} results saved to: {csv_path}")
    display(results_df.groupby('Ticker')[['R2', 'MAE']].agg(['mean', 'std']))



############################################################
STARTING V2 ATTN EXPERIMENTS | SEQUENCE LENGTH: 5
############################################################

>>> Processing: MSFT (Seq: 5)
  Run 01: R2=0.9776
  Run 02: R2=0.9687
  Run 03: R2=0.9693
  Run 04: R2=0.9583
  Run 05: R2=0.9736
  Run 06: R2=0.9574
  Run 07: R2=0.9412
  Run 08: R2=0.9790
  Run 09: R2=0.9769
  Run 10: R2=0.9714

>>> Processing: DIS (Seq: 5)
  Run 01: R2=0.8811
  Run 02: R2=0.8593
  Run 03: R2=0.8871
  Run 04: R2=0.8762
  Run 05: R2=0.9185
  Run 06: R2=0.8336
  Run 07: R2=0.8864
  Run 08: R2=0.9095
  Run 09: R2=0.9359
  Run 10: R2=0.8846

>>> Processing: WMT (Seq: 5)
  Run 01: R2=0.8792
  Run 02: R2=0.8815
  Run 03: R2=0.9237
  Run 04: R2=0.9463
  Run 05: R2=0.9442
  Run 06: R2=0.9400
  Run 07: R2=0.9057
  Run 08: R2=0.8525
  Run 09: R2=0.9558
  Run 10: R2=0.8484

[DONE] Seq 5 results saved to: results/trainV2_attention\v2_att_stability_results_5.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.887214  0.029175  0.018082  0.002817
MSFT    0.967335  0.011813  0.022986  0.004418
WMT     0.907719  0.040149  0.028379  0.007676


############################################################
STARTING V2 ATTN EXPERIMENTS | SEQUENCE LENGTH: 10
############################################################

>>> Processing: MSFT (Seq: 10)
  Run 01: R2=0.9754
  Run 02: R2=0.8826
  Run 03: R2=0.9701
  Run 04: R2=0.9710
  Run 05: R2=0.9632
  Run 06: R2=0.9680
  Run 07: R2=0.9764
  Run 08: R2=0.9771
  Run 09: R2=0.9748
  Run 10: R2=0.9269

>>> Processing: DIS (Seq: 10)
  Run 01: R2=0.9150
  Run 02: R2=0.8865
  Run 03: R2=0.9266
  Run 04: R2=0.9087
  Run 05: R2=0.9244
  Run 06: R2=0.9117
  Run 07: R2=0.9283
  Run 08: R2=0.9103
  Run 09: R2=0.8816
  Run 10: R2=0.9130

>>> Processing: WMT (Seq: 10)
  Run 01: R2=0.9069
  Run 02: R2=0.7847
  Run 03: R2=0.9260
  Run 04: R2=0.9358
  Run 05: R2=0.9410
  Run 06: R2=0.9422
  Run 07: R2=0.9080
  Run 08: R2=0.8579
  Run 09: R2=0.9242
  Run 10: R2=0.8430

[DONE] Seq 10 results saved to: results/trainV2_attention\v2_att_stability_results_10.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.910592  0.015677  0.016064  0.001401
MSFT    0.958558  0.030513  0.025087  0.009314
WMT     0.896974  0.052017  0.029887  0.009220


############################################################
STARTING V2 ATTN EXPERIMENTS | SEQUENCE LENGTH: 20
############################################################

>>> Processing: MSFT (Seq: 20)
  Run 01: R2=0.9640
  Run 02: R2=0.9173
  Run 03: R2=0.8847
  Run 04: R2=0.9502
  Run 05: R2=0.9652
  Run 06: R2=0.9670
  Run 07: R2=0.9379
  Run 08: R2=0.8884
  Run 09: R2=0.9711
  Run 10: R2=0.9591

>>> Processing: DIS (Seq: 20)
  Run 01: R2=0.8739
  Run 02: R2=0.9032
  Run 03: R2=0.8909
  Run 04: R2=0.8827
  Run 05: R2=0.8850
  Run 06: R2=0.8829
  Run 07: R2=0.9071
  Run 08: R2=0.8835
  Run 09: R2=0.8411
  Run 10: R2=0.7926

>>> Processing: WMT (Seq: 20)
  Run 01: R2=0.9196
  Run 02: R2=0.9421
  Run 03: R2=0.8828
  Run 04: R2=0.9403
  Run 05: R2=0.8238
  Run 06: R2=0.8114
  Run 07: R2=0.9210
  Run 08: R2=0.9336
  Run 09: R2=0.9151
  Run 10: R2=0.8196

[DONE] Seq 20 results saved to: results/trainV2_attention\v2_att_stability_results_20.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.874287  0.033837  0.017828  0.002276
MSFT    0.940492  0.032684  0.028349  0.008724
WMT     0.890926  0.052883  0.029674  0.008764